In [11]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [12]:
# Đọc dữ liệu
df = pd.read_csv('./products.csv')
df.head(3)

,Product_ID,name,main_category,sub_category,ratings,no_of_ratings,PRICE
0,1,"Redmi 10 Power (Power Black, 8GB RAM, 128GB St...",Electronics,Electronic Gadgets,4.0,965,AUD 358.47
1,2,"OnePlus Nord CE 2 Lite 5G (Blue Tide, 6GB RAM,...",Electronics,Electronic Gadgets,4.3,226,AUD 377.34
2,3,OnePlus Bullets Z2 Bluetooth Wireless in Ear E...,Electronics,Electronic Gadgets,4.2,54,AUD 43.38


In [13]:
import re
# Function to clean the product names
def clean_product_name(name):
    name = re.sub(r'[\(\)\|,:"&\-]', '', name)  # remove special characters 
    name = re.sub(r'\s+', ' ', name)  
    return name.strip()  

df['cleaned_name'] = df['name'].apply(clean_product_name)
df[['name', 'cleaned_name']].head(10)

,name,cleaned_name
0,"Redmi 10 Power (Power Black, 8GB RAM, 128GB St...",Redmi 10 Power Power Black 8GB RAM 128GB Storage
1,"OnePlus Nord CE 2 Lite 5G (Blue Tide, 6GB RAM,...",OnePlus Nord CE 2 Lite 5G Blue Tide 6GB RAM 12...
2,OnePlus Bullets Z2 Bluetooth Wireless in Ear E...,OnePlus Bullets Z2 Bluetooth Wireless in Ear E...
3,"Samsung Galaxy M33 5G (Mystique Green, 6GB, 12...",Samsung Galaxy M33 5G Mystique Green 6GB 128GB...
4,"OnePlus Nord CE 2 Lite 5G (Black Dusk, 6GB RAM...",OnePlus Nord CE 2 Lite 5G Black Dusk 6GB RAM 1...
5,"Redmi 10 Power (Sporty Orange, 8GB RAM, 128GB ...",Redmi 10 Power Sporty Orange 8GB RAM 128GB Sto...
6,boAt Airdopes 141 Bluetooth Truly Wireless in ...,boAt Airdopes 141 Bluetooth Truly Wireless in ...
7,"Apple 20W USB-C Power Adapter (for iPhone, iPa...",Apple 20W USBC Power Adapter for iPhone iPad A...
8,"Fire-Boltt Ninja Call Pro Plus 1.83"" Smart Wat...",FireBoltt Ninja Call Pro Plus 1.83 Smart Watch...
9,"Samsung Galaxy M33 5G (Emerald Brown, 6GB, 128...",Samsung Galaxy M33 5G Emerald Brown 6GB 128GB ...


In [14]:
df['product_details'] = (
    df['name'] + ' ' + 
    df['main_category'] + ' ' + 
    df['sub_category'] 
)

In [15]:
# Tính toán TF-IDF
tfidf_vec = TfidfVectorizer(stop_words='english', max_df=0.95, min_df=2, ngram_range=(1, 1))
tfidf_matrix = tfidf_vec.fit_transform(df['product_details'])

In [16]:
tfidf_matrix.shape

(50000, 13168)

In [17]:
# Tính toán K-NN
knn = NearestNeighbors(n_neighbors=5, metric='cosine')  # Sử dụng cosine distance
knn.fit(tfidf_matrix)

NearestNeighbors(metric='cosine')

In [19]:
def content_based_knn_recommendations(product_id, data, knn_model):
    product_index = data.index[data['Product_ID'] == product_id].tolist()[0]  
    distances, indices = knn_model.kneighbors(tfidf_matrix[product_index], n_neighbors=10)   
    return data.iloc[indices.flatten()[1:]], distances.flatten()[1:]

input_product_id = int(input("Nhập Product_ID: "))
recommendations_result, distances = content_based_knn_recommendations(input_product_id, df, knn)

print("Gợi ý:")
print(recommendations_result[['name']])

Gợi ý:
                                                    name
5      Redmi 10 Power (Sporty Orange, 8GB RAM, 128GB ...
9390    Samsung Galaxy A53 Black, 8GB RAM, 128GB Storage
1646   OnePlus 11 5G (Titan Black, 8GB RAM, 128GB Sto...
9197   Redmi 10 (Caribbean Green, 6GB RAM, 128GB Stor...
1248   OnePlus 11 5G (Eternal Green, 8GB RAM, 128GB S...
3444    Redmi 10 (Midnight Black, 4GB RAM, 64GB Storage)
1885    Redmi 10 (Midnight Black, 4GB RAM, 64GB Storage)
413    OnePlus Nord CE 2 Lite 5G (Black Dusk, 8GB RAM...
18220  In-Ear Headphone For Xiaomi Redmi 10 Power , R...
